# 03 — Feedback Loop Pattern

**Mode:** Offline  
**Level:** Fundamentals  
**Estimated time:** 30 minutes

[Watch the original lesson video](https://www.youtube.com/watch?v=uNXC_XDeVlc)

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## What you will build

A deterministic generator–critic loop that revises a draft until a clear termination condition is satisfied. You will track correlation data across iterations and inspect every evaluation Spore.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, get_events, require_env, setup_notebook,
    show_callout, show_flow, show_json, show_roles, show_spore,
    show_timeline, stage,
)

setup_notebook(3, 'Feedback Loop Pattern')


## Prerequisites and setup

**Course prerequisites:** Complete `course-02-research-pipeline`.

**Execution requirements:** Offline. The text is deterministic so the control pattern is easy to study.

Run the setup cell above first. It configures presentation helpers and a
credential-free lifecycle provider. It does not hide any agent workflow.


## Learning goals

- Choreograph a feedback loop through message types.
- Carry correlation state through every iteration.
- Define a bounded termination condition.
- Observe cascading completion across a loop.


## Mental model

A feedback loop is not a hidden `for` loop. The generator reacts to new or revised work; the critic reacts to drafts. A correlation key identifies the work item across messages. A termination condition prevents endless choreography and determines when an approved result is emitted.


In [ ]:
show_flow(
('Generator', 'produce draft', 'agent'),
('Critic', 'evaluate', 'agent'),
('Revision', 'correlated retry', 'spore'),
('Output', 'approved result', 'agent'),
)


## Try it

We will now assemble the workflow in small steps. Run each cell, then pause at the inspection that follows it.


### Create loop state

The dictionaries and lists are inspection state for this lesson. The topic acts as the correlation key and the maximum attempt count is explicit.


In [ ]:
from praval import agent, broadcast, get_reef, start_agents
from praval.core.reef import reset_reef

reset_reef()
attempts = {}
approved = []
evaluation_spores = []
max_attempts = 3


### What just happened?

The loop now has a bounded policy before any agent runs.

### Why this matters

Termination policy should be data you can review and test, not an accidental property of model behavior.


### Define the generator

The same handler accepts the initial request and later revision messages. It increments correlated state, creates a draft, and asks for evaluation.


In [ ]:
@agent(
    "course-generator",
    responds_to=["draft_request", "revise"],
    auto_broadcast=False,
)
def generator(spore):
    topic = spore.knowledge["topic"]
    attempt = attempts.get(topic, 0) + 1
    attempts[topic] = attempt
    draft = f"{topic}: " + ("clear " * attempt).strip()
    stage("generator", "drafted", f"attempt {attempt}", spore)
    broadcast({
        "type": "evaluate", "topic": topic,
        "attempt": attempt, "draft": draft,
    })


### What just happened?

Every `evaluate` Spore contains the topic, attempt number, and draft needed by the next participant.

### Why this matters

Self-contained messages reduce hidden coupling between agents.


### Define the critic

The critic either requests another revision or emits approval. The branch is the loop's visible termination condition.


In [ ]:
@agent("course-critic", responds_to=["evaluate"], auto_broadcast=False)
def critic(spore):
    evaluation_spores.append(spore)
    attempt = spore.knowledge["attempt"]
    stage("critic", "evaluated", f"attempt {attempt}", spore)
    if attempt < max_attempts:
        broadcast({
            "type": "revise",
            "topic": spore.knowledge["topic"],
            "feedback": "make the message clearer",
        })
    else:
        broadcast({
            "type": "approved", "draft": spore.knowledge["draft"],
            "attempts": attempt,
        })


### What just happened?

The critic does not call the generator. A `revise` Spore routes back through the Reef and starts the next iteration.

### Why this matters

Message-based loops preserve observability and let other participants join the review process later.


### Capture approved output and run

A third agent marks the terminal state. One initial Spore is enough to drive all three attempts.


In [ ]:
@agent("course-feedback-output", responds_to=["approved"], auto_broadcast=False)
def output(spore):
    approved.append(spore.knowledge)
    stage("output", "approved", spore.knowledge["draft"], spore)


start_agents(
    generator, critic, output,
    initial_data={"type": "draft_request", "topic": "Visible agents"},
    channel="course-feedback-loop",
)
reef = get_reef()
assert reef.wait_for_completion(timeout=10)
assert approved[0]["attempts"] == max_attempts
assert len(evaluation_spores) == max_attempts


### What just happened?

The single completion wait covered every revision and stopped after the critic published approval.

### Why this matters

A tested terminal result is stronger evidence than assuming a loop became idle.


In [ ]:
show_json(approved[0], "Approved output")
show_spore(evaluation_spores[1], "Second evaluation Spore")
show_timeline()


## Your turn

Add a second topic and use a `request_id` instead of topic text as the correlation key. Verify both items reach approval without sharing attempt counts.


In [ ]:
# request = {"type": "draft_request", "request_id": "work-002", "topic": "Reefs"}
# Key attempts by request_id and carry request_id in evaluate/revise/approved Spores.


## Common mistake

**Mistake:** Allowing a critic to request revision without a maximum attempt or terminal error.

**Correction:** Define both success and exhaustion outcomes, and test that every request reaches one of them.


<details>
<summary>Under the hood</summary>

Reef completion uses counters that include broadcasts performed inside handlers. Because each revision is enqueued before the current handler finishes, `wait_for_completion()` does not return in the middle of the loop.

</details>


## Recap

- Feedback loops are agent choreography expressed through Spores.
- Correlation data keeps concurrent work items separate.
- A bounded termination rule prevents infinite revision.
- The final output should make loop completion explicit.


## Cleanup

Release every agent, transport, temporary file, or external resource created by this lesson. Cleanup is part of the example, not an afterthought.


In [ ]:
for function in (generator, critic, output):
    function._praval_agent.close()
assert reef.shutdown(timeout=10)


### Next lesson

Continue to `04_parallel_agents.ipynb` to fan one request out to specialists and join their independent results safely.
